# Лабораторная работа D2: PCA и kNN на MNIST

## Подготовка данных

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import seaborn as sns

(X_train_full, y_train_full), (X_test_full, y_test_full) = mnist.load_data()
X_train_full = X_train_full[..., None]
X_test_full = X_test_full[..., None]

print("Обучающая выборка:", X_train_full.shape)
print("Тестовая выборка:", X_test_full.shape)
print("Размер одного изображения: 28x28, признаков:", 28*28)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit, ax in enumerate(axes.flat):
    idx = np.where(y_train_full == digit)[0][0]
    ax.imshow(X_train_full[idx].reshape(28, 28), cmap="gray")
    ax.set_title(str(digit))
    ax.axis("off")
plt.tight_layout()
plt.show()

X_all = np.concatenate([X_train_full, X_test_full]).reshape(-1, 784).astype(np.float32) / 255.0
y_all = np.concatenate([y_train_full, y_test_full])
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, train_size=6000, test_size=1500, random_state=42, stratify=y_all
)
print("X_train:", X_train.shape, "X_test:", X_test.shape)

## Метод главных компонент

In [ ]:
class my_PCA:
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.components_ = None
        self.explained_variance_ = None
        self.mean_ = None

    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        variance = S**2 / (X.shape[0] - 1)
        k = len(variance) if self.n_components is None else min(self.n_components, len(variance))
        self.components_ = Vt[:k]
        self.explained_variance_ = variance[:k]
        return self

    def transform(self, X):
        return (X - self.mean_) @ self.components_.T

    def fit_transform(self, X):
        return self.fit(X).transform(X)

pca_check = my_PCA(n_components=50)
sk_pca = PCA(n_components=50)
Y_my = pca_check.fit_transform(X_train[:1000])
Y_sk = sk_pca.fit_transform(X_train[:1000])
print("Сумма собственных значений my_PCA:", round(pca_check.explained_variance_.sum(), 4))
print("Сумма собственных значений sklearn:", round(sk_pca.explained_variance_.sum(), 4))
print("Размеры:", Y_my.shape, Y_sk.shape)

In [ ]:
pca_full = my_PCA()
pca_full.fit(X_train)
cumulative = np.cumsum(pca_full.explained_variance_) / np.sum(pca_full.explained_variance_)
eigenvalues = pca_full.explained_variance_

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(np.arange(1, len(eigenvalues)+1), eigenvalues)
axes[0].set_xlabel("Номер компоненты")
axes[0].set_ylabel("Собственное значение")
axes[0].set_title("Собственные значения")
axes[1].plot(np.arange(1, len(cumulative)+1), cumulative)
axes[1].axhline(0.95, linestyle="--", label="95%")
axes[1].axhline(0.99, linestyle="--", label="99%")
axes[1].set_xlabel("Число компонент")
axes[1].set_ylabel("Доля дисперсии")
axes[1].set_title("Накопленная дисперсия")
axes[1].legend()
plt.tight_layout()
plt.show()

k95 = int(np.searchsorted(cumulative, 0.95) + 1)
print(f"Первые 15 компонент объясняют {cumulative[14]:.3f} дисперсии")
print(f"Для объяснения 95% дисперсии нужно {k95} компонент")
print("Собственные значения показывают дисперсию данных вдоль соответствующих главных компонент.")

In [ ]:
pca_2d = my_PCA(n_components=2)
Y_2d = pca_2d.fit_transform(X_train)
plt.figure(figsize=(9, 7))
for digit in range(10):
    mask = y_train == digit
    plt.scatter(Y_2d[mask, 0], Y_2d[mask, 1], label=str(digit), alpha=0.5, s=10)
plt.legend(title="Цифра", ncol=2)
plt.xlabel("Первая главная компонента")
plt.ylabel("Вторая главная компонента")
plt.title("MNIST в первых двух главных компонентах")
plt.show()
print("Классы частично пересекаются: похожие по написанию цифры трудно разделить только по двум компонентам.")

## Реализация kNN

In [ ]:
class my_kNN:
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=np.float32)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        preds = []
        train_sq = np.sum(self.X_train**2, axis=1)
        for start in range(0, len(X), 100):
            batch = X[start:start+100]
            dists = np.sum(batch**2, axis=1)[:, None] + train_sq[None, :] - 2 * batch @ self.X_train.T
            nearest = np.argpartition(dists, self.n_neighbors-1, axis=1)[:, :self.n_neighbors]
            for ids in nearest:
                counts = np.bincount(self.y_train[ids].astype(int))
                preds.append(np.argmax(counts))
        return np.array(preds)

knn_my = my_kNN(5).fit(X_train, y_train)
y_pred_my = knn_my.predict(X_test)
accuracy_my = accuracy_score(y_test, y_pred_my)
print(f"Точность my_kNN: {accuracy_my:.4f}")

knn_sk = KNeighborsClassifier(n_neighbors=5)
knn_sk.fit(X_train, y_train)
y_pred_sk = knn_sk.predict(X_test)
accuracy_sk = accuracy_score(y_test, y_pred_sk)
print(f"Точность sklearn kNN: {accuracy_sk:.4f}")
print(f"Разница: {abs(accuracy_my-accuracy_sk):.4f}")

## Зависимость от числа соседей

In [ ]:
k_values = [1, 3, 5, 7, 10, 15, 20, 30]
accuracies = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    accuracies.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies, "o-")
plt.xlabel("Число соседей k")
plt.ylabel("Точность")
plt.title("Зависимость точности kNN от числа соседей")
plt.grid(True)
plt.show()
best_k = k_values[int(np.argmax(accuracies))]
print(f"Максимальная точность получена при k={best_k}: {max(accuracies):.4f}")
print("При малом k модель чувствительна к шуму, а при слишком большом соседей учитываются удалённые объекты.")

## PCA и kNN

In [ ]:
n_components_list = [1, 2, 5, 10, 15, 20, 30, 50, 75, 100, 200, 300, 500, 784]
accuracies_pca = []
times = []
max_comp = max(n_components_list)
pca_exp = PCA(n_components=max_comp, svd_solver="full", random_state=42)
start = time.time()
X_train_pca_all = pca_exp.fit_transform(X_train)
X_test_pca_all = pca_exp.transform(X_test)
pca_fit_time = time.time() - start

for n_comp in n_components_list:
    start = time.time()
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(X_train_pca_all[:, :n_comp], y_train)
    pred = model.predict(X_test_pca_all[:, :n_comp])
    accuracies_pca.append(accuracy_score(y_test, pred))
    times.append(time.time() - start)

plt.figure(figsize=(9, 5))
plt.plot(n_components_list, accuracies_pca, "o-")
plt.axhline(accuracy_sk, linestyle="--", label=f"kNN без PCA ({accuracy_sk:.3f})")
plt.xlabel("Число компонент")
plt.ylabel("Точность")
plt.title("Точность kNN после PCA")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(n_components_list, times, "o-")
plt.xlabel("Число компонент")
plt.ylabel("Время обучения и предсказания, сек")
plt.title("Время работы kNN в зависимости от числа компонент")
plt.grid(True)
plt.show()

best_comp = n_components_list[int(np.argmax(accuracies_pca))]
print(f"Наилучшая точность PCA+kNN: {max(accuracies_pca):.4f}, компонент: {best_comp}")
print(f"Время вычисления PCA: {pca_fit_time:.2f} сек")
print("При уменьшении числа компонент kNN обычно работает быстрее, так как расстояния считаются по меньшему числу признаков.")

## Подбор параметров

In [ ]:
n_components_grid = [5, 10, 15, 20, 30, 50, 100]
k_grid = [1, 3, 5, 7, 10]
accuracy_matrix = np.zeros((len(n_components_grid), len(k_grid)))

for i, n_comp in enumerate(n_components_grid):
    for j, k in enumerate(k_grid):
        model = KNeighborsClassifier(n_neighbors=k)
        model.fit(X_train_pca_all[:, :n_comp], y_train)
        pred = model.predict(X_test_pca_all[:, :n_comp])
        accuracy_matrix[i, j] = accuracy_score(y_test, pred)

plt.figure(figsize=(9, 5))
sns.heatmap(accuracy_matrix, annot=True, fmt=".3f",
            xticklabels=k_grid, yticklabels=n_components_grid, cmap="viridis")
plt.xlabel("Число соседей k")
plt.ylabel("Число компонент PCA")
plt.title("Точность классификации MNIST")
plt.show()

best_idx = np.unravel_index(np.argmax(accuracy_matrix), accuracy_matrix.shape)
worst_idx = np.unravel_index(np.argmin(accuracy_matrix), accuracy_matrix.shape)
print("Лучшая комбинация: компонент =", n_components_grid[best_idx[0]],
      ", k =", k_grid[best_idx[1]], ", accuracy =", round(accuracy_matrix[best_idx], 4))
print("Худшая комбинация: компонент =", n_components_grid[worst_idx[0]],
      ", k =", k_grid[worst_idx[1]], ", accuracy =", round(accuracy_matrix[worst_idx], 4))

## Вывод

In [ ]:
print("1. PCA уменьшает число признаков, оставляя направления с наибольшей дисперсией.")
print("2. kNN относит объект к наиболее частому классу среди ближайших объектов; он прост, но может быть медленным.")
print("3. PCA уменьшает время классификации, но при слишком малом числе компонент теряется важная информация.")
print("4. Оптимальное число компонент определяется экспериментом и зависит от выбранной метрики качества.")
print("5. Самым сложным было реализовать вычисление расстояний и подобрать параметры.")
print("6. Самым интересным было сравнить точность на исходных данных и после PCA.")